# PARIKSHAN — Exploratory Data Analysis

> ⚠️ **SYNTHETIC DATA.** This notebook explores the generated CUF-style panel
> described in `docs/DATA_METHODOLOGY.md`. Every statistic below reflects the
> generator's calibration, not real Indian infrastructure outcomes — see PRD
> §4.3 for why that distinction matters and how this project uses these
> results (methodology validation, not a real-world accuracy claim).

Covers five analyses (PRD §7/Phase 2 task 4):
1. Distributions of cost, duration, and overrun outcomes
2. Sector x cost-band overrun heatmap
3. Delay-reason co-occurrence
4. `progress_gap_pp` as an early leading indicator of terminal cost overrun
5. Right-censoring analysis


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter

from paimana import config
from paimana.data.loader import load_processed_panel, load_processed_projects

plt.rcParams["figure.dpi"] = 100

panel = load_processed_panel()
projects = load_processed_projects()
completed = projects[projects["is_censored"] == False].copy()

print(f"panel: {panel.shape}, projects: {projects.shape}, completed: {completed.shape}")


## 1. Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].hist(projects["original_cost_cr"].clip(upper=projects["original_cost_cr"].quantile(0.99)), bins=40, color="#3b6ea5")
axes[0, 0].set_title("Approved cost (Rs crore, 99th pct clipped)")

axes[0, 1].hist(projects["original_duration_months"], bins=40, color="#3b6ea5")
axes[0, 1].set_title("Original planned duration (months)")

axes[1, 0].hist(completed["cost_overrun_pct"].clip(-20, 150), bins=50, color="#c0504d")
axes[1, 0].axvline(0, color="black", linewidth=0.8)
axes[1, 0].set_title("Cost overrun % (completed projects, clipped at 150%)")

axes[1, 1].hist(completed["time_overrun_months"].clip(-20, 150), bins=50, color="#c0504d")
axes[1, 1].axvline(0, color="black", linewidth=0.8)
axes[1, 1].set_title("Time overrun (months, completed projects, clipped at 150)")

plt.tight_layout()
plt.show()

print(completed[["cost_overrun_pct", "time_overrun_months"]].describe())


## 2. Sector x cost-band overrun heatmap

In [ ]:
pivot = completed.pivot_table(
    index="sector", columns="cost_band", values="cost_overrun_pct", aggfunc="mean"
)
cost_band_order = ["150-500", "500-1000", "1000-5000", ">5000"]
pivot = pivot.reindex(columns=[c for c in cost_band_order if c in pivot.columns])
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(pivot.values, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_title("Mean cost overrun % by sector x cost band (completed projects)")
fig.colorbar(im, ax=ax, label="mean cost overrun %")
plt.tight_layout()
plt.show()


## 3. Delay-reason co-occurrence

In [ ]:
reason_cols = [f"reason_{r}" for r in config.DELAY_REASONS]
active = panel[panel["n_reasons_active"] > 0][reason_cols].astype(int)

cooccur = active.T.dot(active)
np.fill_diagonal(cooccur.values, 0)  # zero the diagonal so the heatmap contrast isn't dominated by self-counts

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cooccur.values, cmap="viridis")
labels = [r.replace("reason_", "") for r in reason_cols]
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=7)
ax.set_title("Delay-reason co-occurrence (quarters both active, diagonal zeroed)")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 4. `progress_gap_pp` as an early leading indicator of terminal cost overrun

In [ ]:
# Early-quarter snapshot per project: the row closest to 25% of planned duration elapsed.
panel_completed = panel[panel["is_censored"] == False].copy()
panel_completed["dist_to_quarter_mark"] = (panel_completed["elapsed_frac"] - 0.25).abs()
early_snapshot = (
    panel_completed.sort_values("dist_to_quarter_mark")
    .groupby("project_id", as_index=False)
    .first()
)

merged = early_snapshot[["project_id", "progress_gap_pp"]].merge(
    completed[["project_id", "cost_overrun_pct"]], on="project_id"
)
corr = merged["progress_gap_pp"].corr(merged["cost_overrun_pct"])
print(f"Correlation (early progress_gap_pp vs. terminal cost_overrun_pct): {corr:.3f}")

merged["gap_quartile"] = pd.qcut(merged["progress_gap_pp"], 4, labels=["Q1 (lowest gap)", "Q2", "Q3", "Q4 (highest gap)"])
binned = merged.groupby("gap_quartile", observed=True)["cost_overrun_pct"].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(merged["progress_gap_pp"], merged["cost_overrun_pct"], alpha=0.25, s=10)
axes[0].set_xlabel("progress_gap_pp at ~25% elapsed")
axes[0].set_ylabel("terminal cost_overrun_pct")
axes[0].set_title(f"r = {corr:.3f}")

binned.plot(kind="bar", ax=axes[1], color="#3b6ea5")
axes[1].set_ylabel("mean terminal cost_overrun_pct")
axes[1].set_title("Terminal overrun by early spending-gap quartile")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 5. Right-censoring analysis

In [ ]:
censor_by_sector = projects.groupby("sector")["is_censored"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 8))
censor_by_sector.plot(kind="barh", ax=ax, color="#8064a2")
ax.set_xlabel("share still ongoing (right-censored)")
ax.set_title("Right-censoring share by sector")
plt.tight_layout()
plt.show()

print(f"Portfolio-wide censored share: {projects['is_censored'].mean():.1%}")

# Kaplan-Meier: time-to-completion, right-censored for still-ongoing projects.
kmf = KaplanMeierFitter()
kmf.fit(
    durations=projects["months_to_completion"],
    event_observed=~projects["is_censored"],
    label="Portfolio",
)
fig, ax = plt.subplots(figsize=(7, 5))
kmf.plot_survival_function(ax=ax)
ax.set_xlabel("Months since sanction")
ax.set_ylabel("Share not yet completed")
ax.set_title("Kaplan-Meier: time to project completion")
plt.tight_layout()
plt.show()


## Summary

- Cost and time overrun distributions are right-skewed by design (PRD §4.3.5,
  `docs/DATA_METHODOLOGY.md` §6): most completed projects finish near budget,
  a risk-weighted minority drives the portfolio average up.
- Sector and cost-band both visibly modulate overrun risk — exactly the
  structure Phase 3/4 models need to learn from static attributes.
- Delay reasons co-occur rather than firing independently, supporting the
  `n_reasons_active` feature as a meaningful risk signal.
- `progress_gap_pp`, measured as early as ~25% into a project's planned
  duration, is a genuine (if noisy) leading indicator of terminal cost
  overrun — evidence the modelling task is learnable without being trivial.
- Right-censoring is concentrated among more recently-sanctioned projects,
  consistent with the generator's mechanism (PRD §4.4 methodology §7) rather
  than an artificial independent flag.
